# ADAPT Smoke Training Notebook

This notebook runs the repo's local smoke training flow on a Google Colab or Kaggle GPU runtime.

What it does:
- detects Colab vs Kaggle
- clones or reuses the repo locally
- installs the repo with training extras
- validates the GPU and Python training stack
- runs the existing smoke training entrypoint from `training/train_grpo.py`
- surfaces artifacts and failure logs for low-cost debugging

What it does not do:
- no Hugging Face Space deploy
- no model repo upload
- no `/train` or `/train/status` API usage

Recommended runtime:
- GPU enabled
- Colab T4/L4 or Kaggle P100/T4 is fine for a smoke run

In [ ]:
import os
import platform
from pathlib import Path

IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
IS_KAGGLE = Path("/kaggle/working").exists()

print({
    "platform": platform.platform(),
    "python": platform.python_version(),
    "is_colab": IS_COLAB,
    "is_kaggle": IS_KAGGLE,
})

if not (IS_COLAB or IS_KAGGLE):
    print("This notebook is optimized for Colab or Kaggle, but it can still run elsewhere.")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/s-shah4/meta-rl-dsa-solver.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/meta-rl-dsa-solver") if IS_COLAB else Path("/kaggle/working/meta-rl-dsa-solver") if IS_KAGGLE else Path.cwd()

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    print(f"Reusing existing repo at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

In [ ]:
import os
import shlex
import subprocess
import sys

install_commands = [
    "apt-get update",
    "apt-get install -y build-essential",
    f"{shlex.quote(sys.executable)} -m pip install --upgrade pip setuptools wheel",
    f"{shlex.quote(sys.executable)} -m pip install -e '.[train]'",
]

for command in install_commands:
    print(f"\\n>>> {command}")
    subprocess.run(command, shell=True, check=True)

os.environ.setdefault("CC", "gcc")
os.environ.setdefault("CXX", "g++")
print({"CC": os.environ.get("CC"), "CXX": os.environ.get("CXX")})

In [ ]:
import importlib
import json
import subprocess
import sys

import torch

print("Python executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
    subprocess.run(["nvidia-smi"], check=False)
else:
    raise RuntimeError("A GPU runtime is required for this smoke notebook.")

versions = {}
for module_name in ["transformers", "trl", "unsloth", "peft"]:
    module = importlib.import_module(module_name)
    versions[module_name] = getattr(module, "__version__", "unknown")

print(json.dumps(versions, indent=2))

In [ ]:
from pathlib import Path

PRESET = "smoke"
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
NOTEBOOK_OUTPUT_ROOT = Path("/content/adapt-notebook-outputs") if IS_COLAB else Path("/kaggle/working/adapt-notebook-outputs") if IS_KAGGLE else Path.cwd() / "adapt-notebook-outputs"
OUTPUT_DIR = NOTEBOOK_OUTPUT_ROOT / PRESET

# Smoke defaults that stay aligned with training/train_grpo.py.
DISABLE_WANDB = True
BASELINE_EVAL = False
TRACE_LOGGING_ENABLED = True

# Optional debugging overrides. Leave as None to preserve preset behavior.
DATASET_SIZE_OVERRIDE = None
MAX_STEPS_OVERRIDE = None
NUM_GENERATIONS_OVERRIDE = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({
    "preset": PRESET,
    "model_name": MODEL_NAME,
    "output_dir": str(OUTPUT_DIR),
    "disable_wandb": DISABLE_WANDB,
    "baseline_eval": BASELINE_EVAL,
    "trace_logging_enabled": TRACE_LOGGING_ENABLED,
})

In [ ]:
import contextlib
import json
import traceback
from pathlib import Path

from training.train_grpo import build_training_config, run_training

overrides = {
    "model_name": MODEL_NAME,
    "output_dir": str(OUTPUT_DIR),
    "disable_wandb": DISABLE_WANDB,
    "baseline_eval": BASELINE_EVAL,
    "trace_logging_enabled": TRACE_LOGGING_ENABLED,
}
if DATASET_SIZE_OVERRIDE is not None:
    overrides["dataset_size"] = DATASET_SIZE_OVERRIDE
if MAX_STEPS_OVERRIDE is not None:
    overrides["max_steps"] = MAX_STEPS_OVERRIDE
if NUM_GENERATIONS_OVERRIDE is not None:
    overrides["num_generations"] = NUM_GENERATIONS_OVERRIDE

config = build_training_config(preset=PRESET, overrides=overrides)
LOG_PATH = Path(config.output_dir) / "notebook_train.log"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

training_summary = None
training_error = None
training_traceback = None

print("Launching smoke training with config:")
print(json.dumps(config.to_dict(), indent=2))
print(f"Notebook log path: {LOG_PATH}")

with LOG_PATH.open("w", encoding="utf-8") as log_handle:
    with contextlib.redirect_stdout(log_handle), contextlib.redirect_stderr(log_handle):
        try:
            training_summary = run_training(config)
        except Exception as exc:
            training_error = str(exc)
            training_traceback = traceback.format_exc()
            print(training_traceback)

if training_summary is not None:
    print("Smoke training completed successfully.")
    print(json.dumps(training_summary, indent=2))
else:
    print("Smoke training failed.")
    print(training_error)
    print(f"Inspect the full log at: {LOG_PATH}")

In [ ]:
import json
from pathlib import Path

artifact_dir = Path(config.output_dir)
print(f"Artifact directory exists: {artifact_dir.exists()}")
print(f"Artifact directory: {artifact_dir}")

if artifact_dir.exists():
    for path in sorted(artifact_dir.rglob("*")):
        if path.is_file():
            print(path.relative_to(artifact_dir), path.stat().st_size)

reward_curve_path = artifact_dir / "reward_curve.csv"
trace_dir = artifact_dir / "logs"

print({
    "reward_curve_exists": reward_curve_path.exists(),
    "trace_dir_exists": trace_dir.exists(),
})

if training_summary is not None:
    print("\\nFinal summary payload:")
    print(json.dumps(training_summary, indent=2))

In [ ]:
from pathlib import Path

log_path = Path(LOG_PATH)
print(f"Log path: {log_path}")

if log_path.exists():
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    tail = lines[-200:]
    print("\\nLast 200 log lines:\\n")
    print("\\n".join(tail))

if training_summary is None:
    print("\\nFailure triage:")
    print("- inspect notebook_train.log for the full traceback")
    print("- inspect the precision audit or import errors in the log output")
    print("- inspect any files under the local output directory, especially logs/ if present")
    if training_traceback:
        print("\\nCaptured traceback:\\n")
        print(training_traceback)